## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: June 27, 2025

# Purpose: This notebook performs various tasks labeled "Exercises" for Week 3/4 of the class. Data must be present within the folder that this notebook is run in.
# This week focuses on Packages/Data Formats/Data Structures/Data Sources

In [18]:
import pandas as pd

### Load Data


In [19]:
df = pd.read_csv("air_quality_by_state_county.csv")
print(df.head())
print(df.shape)


   index  MeasureId                                        MeasureName  \
0      0         83  Number of days with maximum 8-hour average ozo...   
1      1         83  Number of days with maximum 8-hour average ozo...   
2      2         83  Number of days with maximum 8-hour average ozo...   
3      3         83  Number of days with maximum 8-hour average ozo...   
4      4         83  Number of days with maximum 8-hour average ozo...   

  MeasureType StratificationLevel  StateFips StateName  CountyFips CountyName  \
0      Counts      State x County          1   Alabama        1051     Elmore   
1      Counts      State x County          1   Alabama        1073  Jefferson   
2      Counts      State x County          1   Alabama        1079   Lawrence   
3      Counts      State x County          1   Alabama        1089    Madison   
4      Counts      State x County          1   Alabama        1097     Mobile   

   ReportYear  Value      Unit  UnitName    DataOrigin  MonitorOnly 

# Do 5 data transformations

### Convert Value to a consistent numeric format

In [20]:
# Converts Value column to numeric, handling errors by coercing them to NaN
# This is important for ensuring that the data can be used for numerical analysis
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

### Drop duplicates

In [21]:
# If the dataset contains duplicate rows, drop them
# This is important to ensure that each row represents a unique observation
df = df.drop_duplicates()

### Make new Column for county and state together

In [22]:
# There are some instances where Counties can match multiple states, so we're going to create unique identifiers
df['StateAndCounty'] = df['CountyName'].str.strip() + '_' + df['StateName'].str.strip()
# After this, we don't need the county name, so we'll drop it
df = df.drop(columns=['CountyName'])

### Drop Columns that are not needed

In [23]:
# The reason I'm dropping fips is because each simple corresponds to a location id. For example, every row with "Alabama" will have the same fips code. This is basically an encoded variable. Because we're already encoding our own variables, we don't need fips
df = df.drop(columns=['StateFips', 'CountyFips'])

# Dropping index because it isn't needed, it's going to present in a DF anyway
df = df.drop(columns=['index'])

# Dropping MeasureId since it's just the encoded version of the MeasureName
df = df.drop(columns=['MeasureId'])

df.columns 

Index(['MeasureName', 'MeasureType', 'StratificationLevel', 'StateName',
       'ReportYear', 'Value', 'Unit', 'UnitName', 'DataOrigin', 'MonitorOnly',
       'StateAndCounty'],
      dtype='object')

### Standardize column names

In [24]:
# List of categorical columns to convert to numeric codes
cols_to_encode = [
    'MeasureName', 'MeasureType', 'StratificationLevel', 'StateName',
    'Unit', 'UnitName', 'DataOrigin', 'MonitorOnly'
]

# Loop through each column and apply factorization
for col in cols_to_encode:
    # pd.factorize converts each unique category to a unique integer
    # It returns a tuple: (array of codes, array of unique values)
    # Only the codes are assigned back to the DataFrame
    df[col], _ = pd.factorize(df[col])

# Display the first 5 rows of the modified DataFrame
print(df.head())

# Print the shape of the DataFrame (rows, columns)
print(df.shape)


   MeasureName  MeasureType  StratificationLevel  StateName  ReportYear  \
0            0            0                    0          0        1999   
1            0            0                    0          0        1999   
2            0            0                    0          0        1999   
3            0            0                    0          0        1999   
4            0            0                    0          0        1999   

   Value  Unit  UnitName  DataOrigin  MonitorOnly     StateAndCounty  
0    5.0     0         0           0            0     Elmore_Alabama  
1   39.0     0         0           0            0  Jefferson_Alabama  
2   28.0     0         0           0            0   Lawrence_Alabama  
3   31.0     0         0           0            0    Madison_Alabama  
4   32.0     0         0           0            0     Mobile_Alabama  
(218635, 11)


1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:
1. What changes were made to the data?
2. Are there any legal or regulatory guidelines for your data or project topic?
3. What risks could be created based on the transformations done?
4. Did you make any assumptions in cleaning/transforming the data?
5. How was your data sourced / verified for credibility?
6. Was your data acquired in an ethical way?
7. How would you mitigate any of the ethical implications you have identified?

My data comprised numerical and categorical data, but most of my data is categories. Therefore, one of my top priorities was encoding data. I soon discovered that some columns such as 'Fips' were basically encoded names of other columns. Thefore, I dropped the State and County fip columns. The same applies to MeasureId. Index was also irrelevant so I dropped it as well. County wasn't a useful column on its own, so I combined it with State to make a new column so the StateAndCounty is unique, to avoid situations where different states had the same county names. Some rows might be duplicates, so I dropped those as well to avoid situations where data counted twice for certain cateogries. Finally, I converted values to a consistent numeric format.

With my changes, I saw 0 data loss, with 218,635 rows at the beginning of the data transformation and the same number at the end. The number of columns dropped from 15 to 11. These data transofmrations were neither risky nor carried any ethical complications. Simply put, the data is cleaner with less redundancy. As mentioned in milestone 1, my data comes from kaggle but their source is the EPA. 

I've achieved my goals with this data transformations, but I might change the transformations when dealing with the other sources (api and website). 
